# Aula 10 — Checkpoint 4 · Release v2.0 — "O Estoque" · Versão B
**Computational Thinking with Python · FIAP Rio · Semestre 2**

---

Hoje não tem conteúdo novo: tem **release**. Esta é a **versão B** do Checkpoint 4 — a mesma v2.0, a mesma rubrica, por outro caminho. Em vez de escrever a release do zero, você **assume uma release que já foi ao ar**, conserta o que quebrou e entrega o que ficou faltando.

- **Rode as células de cima para baixo.** Travou? **Runtime ▸ Restart and run all** (Colab) ou **Run ▸ Run All** (PyCharm/Jupyter).
- Quando aparecer `# TODO`, o código é seu — **inclusive quando a célula já vem com código**: aí o trabalho é consertar, não apagar e recomeçar sem ler.
- As células ✅ **CONFIRA** são a mesa de teste da release: todas têm que terminar em *"Tudo certo!"*.


## A release da vez: v2.0 — "O Estoque"

O caixa v1.0 calculava; a v2.0 **controla**. O que ela precisa conter (a lista é a mesma da versão A — release é release):

1. **Cardápio e estoque em dicionário** — o estoque aninhado `{produto: {"preco": ..., "qtd": ...}}` (Aulas 07–08).
2. **Baixa automática**: vender desconta do estoque na hora, e **ninguém vende o que não tem**.
3. **Filtros por conjunto**: o que está em falta, o que está disponível, o que é seguro para um cliente alérgico (Aula 09).
4. **Funções documentadas**: anotação de tipo e docstring em todas — e nenhuma consulta pode levantar `KeyError` (o `get`/`in` é a defesa; `try/except` só chega na Aula 11).

## A rubrica

| critério | o que o professor olha |
|---|---|
| **Roda** | Restart and run all termina sem traceback |
| **Robusto** | produto inexistente, quantidade indisponível, quantidade zero: o sistema responde, não cai |
| **Asserts** | todas as células ✅ CONFIRA em "Tudo certo" |
| **PEP 8** | nomes minúsculos com underscore, constantes em MAIÚSCULAS, contratos completos |


## O cenário: o caderno de reclamações

A v2.0 da sua lanchonete **foi ao ar na sexta-feira** — um rascunho que um colega montou às pressas (com "ajuda" de IA) antes de sair do time. No sábado, o caderno de reclamações do balcão encheu. Cinco anotações, na letra do gerente:

| # | reclamação | o que está por trás |
|---|---|---|
| 1 | *"Cliente pediu Pizza e o caixa **apagou**."* | consulta que levanta `KeyError` |
| 2 | *"Vendemos 3 açaís que **não tínhamos** — e o estoque nem mexeu."* | baixa automática que não existe |
| 3 | *"Cliente alérgico perguntou o que podia comer e **ninguém soube responder**."* | os filtros por conjunto ficaram de fora |
| 4 | *"O fechamento do caixa **não fecha** — e a contagem de vendidos mente."* | padrão contador escrito errado |
| 5 | *"Quero ver o turno inteiro rodar **sem ninguém segurando a mão do sistema**."* | a mesa de teste completa |

Cada reclamação é uma etapa, e cada etapa tem a sua mesa de teste. Você **herda o código** — ele está aí, rodando, errado. Leia antes de mexer: na defesa oral, a pergunta é *"onde estava o bug?"*, não *"o que você digitou?"*. A regra de sempre vale: **uma função chama a outra, nenhuma conta mora em dois lugares.**

▶️ RODE a célula abaixo: os dados de partida da release.


In [ ]:
# --- dados de partida da v2.0 (fornecidos — não altere os valores) ---
LANCHONETE = "Podrão do João Thees"   # 👉 só aqui é seu — batize!
TAXA_ENTREGA = 5.00

ESTOQUE = {
    "X-Burguer": {"preco": 18.50, "qtd": 8},
    "X-Salada": {"preco": 20.00, "qtd": 4},
    "Batata frita": {"preco": 10.00, "qtd": 12},
    "Açaí 300ml": {"preco": 12.00, "qtd": 0},
    "Suco de laranja": {"preco": 8.00, "qtd": 6},
    "Refrigerante lata": {"preco": 6.50, "qtd": 30},
    "Água mineral": {"preco": 4.00, "qtd": 15},
    "Combo da casa": {"preco": 29.90, "qtd": 5},
}

INGREDIENTES = {
    "X-Burguer": {"pao", "carne", "queijo"},
    "X-Salada": {"pao", "carne", "queijo", "alface", "tomate"},
    "Batata frita": {"batata", "sal"},
    "Açaí 300ml": {"acai", "granola"},
    "Suco de laranja": {"laranja"},
    "Combo da casa": {"pao", "carne", "queijo", "batata", "sal"},   # X-Burguer + Batata frita (o refrigerante não tem ficha)
}

vendas = []   # cada venda registrada: [produto, qtd, valor]

print(f'{LANCHONETE} — v2.0 no ar. Caderno de reclamações aberto: 5 anotações.')


Sua Lanchonete — v2.0 no ar. Caderno de reclamações aberto: 5 anotações.


### Reclamação 1 — "Cliente pediu Pizza e o caixa apagou" (Aulas 07–08)

O que apareceu na tela do caixa na sexta à noite:

```
KeyError: 'Pizza'
```

As duas consultas herdadas abaixo **funcionam quando o produto existe** — e é exatamente por isso que passaram no teste feliz do seu colega. Elas quebram no primeiro pedido fora do cardápio, e nenhuma tem contrato.

🔧 **Conserte** as duas, sem mudar o nome nem a ordem dos parâmetros:

- `preco_de(estoque, produto) -> float` — o preço, ou **0.0** se o produto não existe.
- `tem_estoque(estoque, produto, qtd) -> bool` — o produto existe **e** a quantidade basta? Cheque com `in` **antes** de descer o nível — o `KeyError` pode vir de qualquer colchete. (Existir com `qtd` 0 não é ter estoque.)

Contrato completo nas duas: **anotação de tipo nos parâmetros e no retorno, e docstring** que diga o que acontece com produto inexistente.


In [44]:
# --- código herdado: roda, mas cai com produto fora do cardápio ---
# CORRIGIDO: checa 'in' antes de descer o nível — nunca levanta KeyError

def preco_de(estoque: dict, produto: str) -> float:
    """Devolve o preço do produto no estoque.

    Se o produto não existir no cardápio, devolve 0.0 (não levanta KeyError).
    """
    if produto in estoque:
        return estoque[produto]["preco"]
    return 0.0


def tem_estoque(estoque: dict, produto: str, qtd: int) -> bool:
    """Diz se o produto existe no estoque E tem quantidade suficiente.

    Checa 'in' antes de descer o nível, então produto inexistente devolve
    False sem levantar KeyError. Existir com qtd 0 também é False.
    """
    if produto not in estoque:
        return False
    return estoque[produto]["qtd"] >= qtd


In [45]:
# ✅ CONFIRA — Reclamação 1
assert preco_de.__doc__ and tem_estoque.__doc__, "faltou docstring em uma das funções — a rubrica cobra contrato completo"
assert preco_de.__annotations__.get("return") is float, "anote o retorno de preco_de: -> float"
assert tem_estoque.__annotations__.get("return") is bool, "anote o retorno de tem_estoque: -> bool"
assert abs(preco_de(ESTOQUE, "Combo da casa") - 29.90) < 0.005, "o Combo da casa custa 29.90"
assert abs(preco_de(ESTOQUE, "Pizza") - 0.0) < 0.005, "produto inexistente devolve 0.0 — sem KeyError (in antes do colchete, ou get com padrão)"
assert tem_estoque(ESTOQUE, "X-Salada", 4) is True, "há exatamente 4 X-Saladas — 4 tem que dar True (e True de verdade: a função promete bool)"
assert tem_estoque(ESTOQUE, "X-Salada", 5) is False, "5 já não tem"
assert tem_estoque(ESTOQUE, "Açaí 300ml", 1) is False, "existir com qtd 0 não é ter estoque"
assert tem_estoque(ESTOQUE, "Pizza", 1) is False, "produto inexistente: False, sem queda — cheque com in antes de descer o nível"
print("Tudo certo! Reclamação 1 fechada — as consultas respondem e nunca caem. ✅")


Tudo certo! Reclamação 1 fechada — as consultas respondem e nunca caem. ✅


### Reclamação 2 — "Vendemos 3 açaís que não tínhamos" (o coração da release)

Sábado, 15h: o açaí acabou às 14h, o caixa vendeu mais três e o estoque continuou dizendo zero. O `vender` herdado está abaixo. Ele **chega a chamar** `tem_estoque` — mas olhe **a ordem das linhas**: registra antes de checar, devolve o valor mesmo recusando, e a baixa é a única coisa que ficou atrás do `if`.

🔧 **Reescreva** `vender(estoque, vendas, produto, qtd) -> float` na ordem certa:

1. **Recusa** quando a quantidade é **zero ou negativa**, quando o produto **não existe** ou quando **não há estoque suficiente** — use `tem_estoque`. Recusa = devolve **0.0** e **não altera nada** (nem estoque, nem histórico).
2. **Aceita**: calcula `valor = preco × qtd` (use `preco_de`), **desconta** `qtd` do estoque, registra `[produto, qtd, valor]` em `vendas` e devolve o `valor`.

A docstring diz que a função **altera** o estoque e a lista de vendas recebidos — é decisão de projeto, e declarada. Nenhum `get` novo aqui dentro: a regra do preço e a regra do estoque já moram em `preco_de` e `tem_estoque`.


In [46]:
# --- código herdado: as linhas certas, na ordem errada ---
# CORRIGIDO: cláusula de guarda — checa tudo que recusa ANTES; recusa não mexe em nada

def vender(estoque: dict, vendas: list, produto: str, qtd: int) -> float:
    """Vende qtd unidades do produto e devolve o valor cobrado.

    ALTERA o estoque (desconta a quantidade) e a lista de vendas (registra
    [produto, qtd, valor]) recebidos. Recusa — devolve 0.0 e não altera nada —
    quando qtd <= 0, o produto não existe ou não há estoque suficiente.
    """
    if qtd <= 0:
        return 0.0
    if not tem_estoque(estoque, produto, qtd):
        return 0.0
    valor = preco_de(estoque, produto) * qtd
    estoque[produto]["qtd"] = estoque[produto]["qtd"] - qtd
    vendas.append([produto, qtd, valor])
    return valor


In [47]:
# ✅ CONFIRA — Reclamação 2
assert vender.__doc__, "faltou a docstring — e ela deve avisar que a função altera o estoque e o histórico"
estoque_teste = {"Refrigerante lata": {"preco": 6.50, "qtd": 4}}
vendas_teste = []

v1 = vender(estoque_teste, vendas_teste, "Refrigerante lata", 4)
assert abs(v1 - 26.00) < 0.005, f"4 × 6.50 = 26.00 — o seu deu {v1:.2f}"
assert estoque_teste["Refrigerante lata"]["qtd"] == 0, f"a baixa automática falhou: deveriam restar 0, restam {estoque_teste['Refrigerante lata']['qtd']}"
assert vendas_teste == [["Refrigerante lata", 4, 26.00]], f"a venda tinha que ser registrada como [produto, qtd, valor] — o seu: {vendas_teste}"

v2 = vender(estoque_teste, vendas_teste, "Refrigerante lata", 1)
assert abs(v2 - 0.0) < 0.005, "acabou: devolve 0.0"
assert len(vendas_teste) == 1, "venda recusada não entra no histórico — cheque ANTES de registrar"
assert vender(estoque_teste, vendas_teste, "Pizza", 1) == 0.0, "produto inexistente: 0.0, sem queda"

estoque_teste["Refrigerante lata"]["qtd"] = 4   # a cozinha repôs
assert vender(estoque_teste, vendas_teste, "Refrigerante lata", 0) == 0.0, "quantidade zero não é pedido: recusa"
assert vender(estoque_teste, vendas_teste, "Refrigerante lata", -2) == 0.0, "quantidade negativa não é pedido (nem devolução): recusa"
assert estoque_teste["Refrigerante lata"]["qtd"] == 4, "recusa não mexe no estoque — nem para cima, nem para baixo"
assert len(vendas_teste) == 1, "as três recusas não entram no histórico"
print("Tudo certo! Reclamação 2 fechada — ninguém vende o que não tem, e zero não é pedido. ✅")


Tudo certo! Reclamação 2 fechada — ninguém vende o que não tem, e zero não é pedido. ✅


### Reclamação 3 — "Cliente alérgico perguntou o que podia comer" (Aula 09)

Nada herdado aqui: os filtros por conjunto **nunca foram escritos**. Três funções, todas com contrato:

- `em_falta(estoque) -> set` — os produtos com `qtd` **igual a 0** (um `for` sobre `estoque.items()` com `add` — ou uma set comprehension).
- `disponiveis(estoque) -> set` — todo mundo **menos** os em falta. **Sem laço**: é uma **diferença** de conjuntos entre `set(estoque)` (as chaves) e `em_falta(estoque)`.
- `cardapio_para(ingredientes, estoque, alergias) -> list[str]` — os produtos **com ficha de ingredientes** que estão **disponíveis** e cuja ficha **não cruza** com as alergias (interseção vazia). Lista **já em ordem alfabética**. A docstring avisa que produto sem ficha (bebida industrializada) fica fora da sugestão — e que produto em falta também.

Repare no **Combo da casa**: ele herda os ingredientes do X-Burguer e da batata. Alérgico a carne não come combo — o conjunto responde isso sozinho, sem `if` especial.


In [48]:
# CORRIGIDO: os três filtros por conjunto da Aula 09, com contrato completo

def em_falta(estoque: dict) -> set:
    """Devolve o conjunto de produtos com quantidade igual a 0 (em falta)."""
    return {produto for produto, dados in estoque.items() if dados["qtd"] == 0}


def disponiveis(estoque: dict) -> set:
    """Devolve o conjunto de produtos disponíveis: todos menos os em falta.
    """
    return set(estoque) - em_falta(estoque)


def cardapio_para(ingredientes: dict, estoque: dict, alergias: set) -> list[str]:
    """Devolve, em ordem alfabética, os produtos seguros para o cliente.
    """
    disponivel = disponiveis(estoque)
    seguros = [
        produto
        for produto, ficha in ingredientes.items()
        if produto in disponivel and ficha & alergias == set()
    ]
    return sorted(seguros)


In [49]:
# ✅ CONFIRA — Reclamação 3
assert em_falta.__doc__ and disponiveis.__doc__ and cardapio_para.__doc__, "faltou docstring em uma das funções"
assert em_falta(ESTOQUE) == {"Açaí 300ml"}, f"só o Açaí está zerado — o seu: {sorted(em_falta(ESTOQUE))}"
assert type(disponiveis(ESTOQUE)) is set, "disponiveis promete set — não lista"
assert disponiveis(ESTOQUE) == set(ESTOQUE) - {"Açaí 300ml"}, "disponível é todo mundo menos quem está em falta — uma diferença de conjuntos"
assert cardapio_para(INGREDIENTES, ESTOQUE, {"carne"}) == ['Batata frita', 'Suco de laranja'], f"alérgico a carne: caem os dois lanches E o combo; o Açaí seria seguro, mas está em falta — o seu: {cardapio_para(INGREDIENTES, ESTOQUE, {'carne'})}"
assert cardapio_para(INGREDIENTES, ESTOQUE, {"tomate", "granola"}) == ['Batata frita', 'Combo da casa', 'Suco de laranja', 'X-Burguer'], "alérgico a tomate e granola: sai a X-Salada (o Açaí já estava em falta) — e a lista vem ordenada"
assert cardapio_para(INGREDIENTES, ESTOQUE, set()) == ['Batata frita', 'Combo da casa', 'Suco de laranja', 'X-Burguer', 'X-Salada'], "sem alergia: todos os disponíveis com ficha — bebida industrializada fica fora"
estoque_reposto = {"Açaí 300ml": {"preco": 12.00, "qtd": 3}, "Água mineral": {"preco": 4.00, "qtd": 0}}
assert cardapio_para(INGREDIENTES, estoque_reposto, {"carne"}) == ['Açaí 300ml'], "o Açaí voltou ao estoque: volta ao cardápio do alérgico — a resposta depende do estoque recebido, não da constante"
print("Tudo certo! Reclamação 3 fechada — o balcão responde ao alérgico com conjuntos, não com if. ✅")


Tudo certo! Reclamação 3 fechada — o balcão responde ao alérgico com conjuntos, não com if. ✅


### Reclamação 4 — "O fechamento não fecha, e a contagem mente" (Aulas 07–08)

O contador herdado está abaixo. 🔮 **PREVEJA** antes de rodar qualquer coisa: para `vendas_exemplo` (a lista da célula CONFIRA), o que ele devolve? Anote no comentário. Dica: ele tem **dois** erros — **um derruba** o programa na primeira venda, **o outro mente** no resultado sem derrubar nada.

🔧 **Conserte** `contagem_por_produto(vendas) -> dict[str, int]` — **unidades vendidas** por produto (o padrão contador da Aula 07, Bloco C: `get` com padrão… e o passo certo).

✍️ **Escreva** `fechar_caixa(vendas) -> float` — o faturamento: a soma da coluna do valor (`sum` sobre um `for` que percorre `vendas`, ou um laço acumulador).

As duas com contrato.


In [ ]:
# 🔮 previsão: para vendas_exemplo, contagem_por_produto devolve ...
# erro que derruba: somando quantidade sem usar o get para pegar cada chave
# erro que mente: se nao tier in da key error

# --- código herdado: dois erros, um traceback ---
# TODO: conserte contagem_por_produto (get com padrão + o passo certo) e escreva o contrato,

def contagem_por_produto(vendas) -> dict[str, int]:
    '''roda vendas e retorna chave de dict contagem somado a quantidade'''
    contagem = {}
    for produto, qtd, valor in vendas:
        
        contagem[produto] = contagem.get(produto, 0) + qtd
        
    return contagem


# TODO: fechar_caixa(vendas) -> float, com contrato — a soma da coluna do valor
def fechar_caixa(vendas: list) -> float:
    '''.'''
    return sum(valor for produto, qtd, valor in vendas)

In [51]:
# ✅ CONFIRA — Reclamação 4
vendas_exemplo = [["X-Burguer", 2, 37.00], ["Água mineral", 3, 12.00], ["X-Burguer", 1, 18.50], ["Combo da casa", 1, 29.90]]
assert contagem_por_produto.__doc__ and fechar_caixa.__doc__, "faltou docstring em uma das funções"
assert contagem_por_produto(vendas_exemplo) == {"X-Burguer": 3, "Água mineral": 3, "Combo da casa": 1}, f"unidades por produto: X-Burguer 3, Água 3, Combo 1 — o seu: {contagem_por_produto(vendas_exemplo)}. O passo é + qtd, não + 1"
assert contagem_por_produto([]) == {}, "sem venda, contagem vazia — e sem KeyError"
assert abs(fechar_caixa(vendas_exemplo) - 97.40) < 0.005, f"37.00 + 12.00 + 18.50 + 29.90 = 97.40 — o seu deu {fechar_caixa(vendas_exemplo):.2f}"
assert abs(fechar_caixa([]) - 0.0) < 0.005, "caixa sem venda fecha em 0.0"
print("Tudo certo! Reclamação 4 fechada — o caixa fecha e a contagem diz a verdade. ✅")


Tudo certo! Reclamação 4 fechada — o caixa fecha e a contagem diz a verdade. ✅


### Reclamação 5 — "Quero ver o turno inteiro rodar" · ✅ a mesa de teste da release

O turno de sábado, de ponta a ponta: sete tentativas de venda — **três aceitas, quatro recusadas** (uma por produto em falta, uma por estoque insuficiente, uma por quantidade zero, uma por produto inexistente). A conta do gerente: 2 × 29.90 + 3 × 4.00 + 3 × 29.90 = **161.50**. E no fim do turno o combo **acabou** — o que o filtro do alérgico tem que refletir sozinho.

Nada para escrever aqui: se a célula terminar em "Release v2.0 conferida", a sua release está de pé. Se não terminar, a mensagem do `assert` diz qual reclamação reabriu.


In [52]:
# ✅ CONFIRA — a mesa de teste completa da release v2.0 (o turno de sábado)
estoque_turno = {
    "Combo da casa": {"preco": 29.90, "qtd": 5},
    "Água mineral": {"preco": 4.00, "qtd": 15},
    "X-Salada": {"preco": 20.00, "qtd": 4},
    "Açaí 300ml": {"preco": 12.00, "qtd": 0},
}
vendas_turno = []

vender(estoque_turno, vendas_turno, "Combo da casa", 2)   # 59.80
vender(estoque_turno, vendas_turno, "Água mineral", 3)    # 12.00
vender(estoque_turno, vendas_turno, "Açaí 300ml", 1)      # recusada: em falta
vender(estoque_turno, vendas_turno, "X-Salada", 5)        # recusada: só há 4
vender(estoque_turno, vendas_turno, "X-Salada", 0)        # recusada: quantidade zero
vender(estoque_turno, vendas_turno, "Combo da casa", 3)   # 89.70 — e o combo acaba
vender(estoque_turno, vendas_turno, "Pizza", 1)           # recusada: não existe

assert len(vendas_turno) == 3, f"três vendas aceitas e quatro recusadas — o seu histórico tem {len(vendas_turno)} (Reclamação 2)"
assert abs(fechar_caixa(vendas_turno) - 161.50) < 0.005, f"59.80 + 12.00 + 89.70 = 161.50 — o seu deu {fechar_caixa(vendas_turno):.2f} (Reclamação 4)"
assert estoque_turno["Combo da casa"]["qtd"] == 0, f"5 - 2 - 3 = 0 combos — o seu estoque diz {estoque_turno['Combo da casa']['qtd']} (Reclamação 2)"
assert estoque_turno["Água mineral"]["qtd"] == 12, "15 - 3 = 12 águas (Reclamação 2)"
assert estoque_turno["X-Salada"]["qtd"] == 4, "as duas recusas da X-Salada não podiam mexer no estoque (Reclamação 2)"
assert contagem_por_produto(vendas_turno) == {"Combo da casa": 5, "Água mineral": 3}, f"unidades por produto: Combo 5, Água 3 — o seu: {contagem_por_produto(vendas_turno)} (Reclamação 4)"
assert em_falta(estoque_turno) == {"Açaí 300ml", "Combo da casa"}, "no fim do turno, Açaí E Combo estão em falta (Reclamação 3)"
assert cardapio_para(INGREDIENTES, estoque_turno, set()) == ['X-Salada'], f"sem alergia, só a X-Salada segue no cardápio: o combo acabou, o açaí não voltou, a água não tem ficha — o seu: {cardapio_para(INGREDIENTES, estoque_turno, set())} (Reclamação 3)"
assert cardapio_para(INGREDIENTES, estoque_turno, {"tomate"}) == [], "alérgico a tomate no fim do turno: nada para oferecer — lista vazia, não erro (Reclamação 3)"
print("Release v2.0 conferida! Cinco reclamações fechadas: consultas que não caem, baixa automática, filtros por conjunto, caixa que fecha. 🎉 ✅")


Release v2.0 conferida! Cinco reclamações fechadas: consultas que não caem, baixa automática, filtros por conjunto, caixa que fecha. 🎉 ✅


## Entrega

- Este notebook completo, com **todas** as células ✅ CONFIRA em "Tudo certo" e o Restart and run all limpo.
- Depois do checkpoint, a **versão de referência da v2.0** fica disponível como código de partida — quem ficou para trás recomeça dela na Aula 11, sem déficit.

**O que vem por aí:** a v2.0 ainda esquece tudo quando o notebook fecha. Na Aula 12 as vendas passam a ser **gravadas em arquivo** — e o sistema começa a ter memória. Antes disso, a Aula 11 traz o `try/except`: a partir de lá, **nenhuma entrada do usuário pode terminar em traceback** — e o `vender` que hoje devolve 0.0 em silêncio vai poder **avisar** por que recusou.
